# SEC MAIS — Teste e Inspeção Bronze: Universidade Para Todos (UPT)

Este notebook inspeciona os dados brutos extraídos do programa **Universidade Para Todos (UPT)** armazenados em `armazenamento/bronze/universidade_para_todos/`.

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Localiza o diretório do UPT na camada bronze
def obter_caminho_upt() -> Path:
    candidatos = [
        Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "universidade_para_todos",
        Path.cwd().parent / "sec-mais" / "armazenamento" / "bronze" / "universidade_para_todos",
        Path.cwd() / "armazenamento" / "bronze" / "universidade_para_todos",
        Path.cwd().parent / "armazenamento" / "bronze" / "universidade_para_todos",
    ]
    for c in candidatos:
        if c.exists():
            return c.resolve()
    return (Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "universidade_para_todos").resolve()

caminho_upt = obter_caminho_upt()
print(f"📁 Caminho UPT: {caminho_upt}")
arquivos = list(caminho_upt.glob("**/*.parquet"))
print(f"📄 Arquivos Parquet encontrados: {len(arquivos)}")
for arq in arquivos:
    print(f"   - {arq.name} ({round(arq.stat().st_size / 1024, 2)} KB)")

## 1. Carregamento dos Dados via Pandas

In [ ]:
if arquivos:
    arquivo_mais_recente = sorted(arquivos, key=lambda f: f.stat().st_mtime, reverse=True)[0]
    print(f"Lendo arquivo: {arquivo_mais_recente.name}")
    
    df = pd.read_parquet(arquivo_mais_recente)
    display(df.head(10))
else:
    print("⚠️ Nenhum arquivo parquet encontrado. Execute a extração primeiro:\npython -m extracao.programas.universidade_para_todos")
    df = pd.DataFrame()

## 2. Estrutura, Tipagem e Verificação de Nulos

In [ ]:
if not df.empty:
    print(f"Total de Linhas: {len(df)}")
    print(f"Total de Colunas: {len(df.columns)}\n")
    
    info_colunas = pd.DataFrame({
        "Tipo": df.dtypes,
        "Valores Não-Nulos": df.notnull().sum(),
        "Nulos": df.isnull().sum(),
        "% Nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "Valores Únicos": df.nunique()
    })
    display(info_colunas)
else:
    print("Sem dados para analisar.")

## 3. Validação de Metadados Técnicos (`_data_extracao`, `_programa`)

In [ ]:
if not df.empty:
    cols_tecnicas = [c for c in df.columns if c.startswith("_")]
    print(f"Colunas técnicas de controle: {cols_tecnicas}")
    if cols_tecnicas:
        display(df[cols_tecnicas].drop_duplicates())
else:
    print("Sem dados para analisar.")

## 4. Consulta SQL Rápida com DuckDB (Opcional)

In [ ]:
try:
    import duckdb
    if arquivos:
        caminho_sql = str(arquivo_mais_recente).replace("\\", "/")
        resultado = duckdb.query(f"SELECT * FROM read_parquet('{caminho_sql}') LIMIT 5").df()
        print("✅ Leitura via DuckDB concluída com sucesso:")
        display(resultado)
except ImportError:
    print("DuckDB não está instalado neste ambiente virtual. (Opcional)")